# 7. Векторизация моделирования фотонов

В этом ноутбуке направления, распространение фотонов, попадания в ФЭУ и фотоэлектронный отклик обрабатываются массивами NumPy. В конце те же этапы сравниваются с циклом по отдельным фотонам. Для честного сравнения используются одинаковые входные направления. Параметры модели: `R=200 см`, `N=40`, `r=10 см`, `a=0.5`, `Y=10000 фотонов/МэВ`.

In [2]:
from math import acos, cos, pi, sin, sqrt
import timeit
import numpy as np

class Source:
    def __init__(self, Y=10000, seed=None):
        self.Y = Y
        self.rng = np.random.default_rng(seed)

    def generate_photons(self, energy):
        if energy < 0:
            raise ValueError("Энергия должна быть неотрицательной")
        photon_count = self.rng.poisson(lam=energy * self.Y)
        return self.generate_directions(photon_count)

    def generate_directions(self, photon_count, rng=None):
        rng = self.rng if rng is None else rng
        cos_theta = rng.uniform(-1.0, 1.0, photon_count)
        phi = rng.uniform(0.0, 2 * np.pi, photon_count)
        sin_theta = np.sqrt(1.0 - cos_theta**2)
        return np.column_stack((
            sin_theta * np.cos(phi),
            sin_theta * np.sin(phi),
            cos_theta
        ))

class Detector:
    def __init__(self):
        self.R = 200.0
        self.N = 40
        self.r = 10.0
        self.a = 0.5
        self.pmt_normals = self._make_pmt_normals()
        self.projection_threshold = np.sqrt(self.R**2 - self.r**2)

    def _make_pmt_normals(self):
        normals = []
        for i in range(self.N):
            phi = acos(1 - 2 * (i + 0.5) / self.N)
            theta = pi * (1 + sqrt(5)) * i
            normals.append((cos(theta) * sin(phi), sin(theta) * sin(phi), cos(phi)))
        return np.asarray(normals, dtype=float)

    def propagate_photons(self, source_position, directions, attenuation_length=None, rng=None):
        position = np.asarray(source_position, dtype=float)
        directions = np.asarray(directions, dtype=float)
        if directions.ndim != 2 or directions.shape[1] != 3:
            raise ValueError("Направления должны иметь форму (число фотонов, 3)")
        norms = np.linalg.norm(directions, axis=1)
        if np.any(norms == 0):
            raise ValueError("Вектор направления должен быть ненулевым")
        directions = directions / norms[:, None]

        position_radius_squared = np.dot(position, position)
        if position_radius_squared >= self.R**2:
            raise ValueError("Источник должен находиться внутри детектора")
        projection = directions @ position
        path_lengths = -projection + np.sqrt(
            projection**2 + self.R**2 - position_radius_squared
        )
        boundary_points = position + path_lengths[:, None] * directions
        pmt_hits = boundary_points @ self.pmt_normals.T >= self.projection_threshold
        has_hit = np.any(pmt_hits, axis=1)
        pmt_indices = np.argmax(pmt_hits, axis=1).astype(np.int64)
        pmt_indices[~has_hit] = -1

        if attenuation_length is not None:
            if attenuation_length <= 0:
                raise ValueError("Длина затухания должна быть положительной")
            rng = np.random.default_rng() if rng is None else rng
            survives = rng.random(len(directions)) < np.exp(-path_lengths / attenuation_length)
            pmt_indices[~survives] = -1

        return pmt_indices, path_lengths

    def generate_photoelectrons(self, pmt_indices, rng=None):
        rng = np.random.default_rng() if rng is None else rng
        pmt_indices = np.asarray(pmt_indices)
        detected = pmt_indices >= 0
        photoelectrons = np.zeros(len(pmt_indices), dtype=np.int64)
        photoelectrons[detected] = rng.poisson(lam=self.a, size=np.count_nonzero(detected))
        return photoelectrons

source = Source(seed=42)
detector = Detector()

In [3]:
energy = 1.0
source_position = np.array([0.0, 0.0, 0.0])
directions = source.generate_photons(energy)
pmt_indices, path_lengths = detector.propagate_photons(
    source_position, directions, attenuation_length=800.0, rng=source.rng
)
photoelectrons = detector.generate_photoelectrons(pmt_indices, rng=source.rng)
charge_by_pmt = np.bincount(
    pmt_indices[pmt_indices >= 0],
    weights=photoelectrons[pmt_indices >= 0],
    minlength=detector.N
).astype(np.int64)

print(f"Сгенерировано фотонов: {len(directions)}")
print(f"Попало на ФЭУ после затухания: {np.count_nonzero(pmt_indices >= 0)}")
print(f"Суммарный заряд: {photoelectrons.sum()} фотоэлектронов")
print(f"Заряд по ФЭУ: {charge_by_pmt}")
print(f"Средний путь до границы: {path_lengths.mean():.1f} см")
assert directions.ndim == 2 and directions.shape[1] == 3
assert pmt_indices.shape == (len(directions),)
assert photoelectrons.shape == (len(directions),)

Сгенерировано фотонов: 10085
Попало на ФЭУ после затухания: 200
Суммарный заряд: 102 фотоэлектронов
Заряд по ФЭУ: [ 1  3  1  2  4  5  3  3  3  5  2  2  1  1  4  1  1  1  4  3  2  3  3  1
  0  0 10  1  4  0  0  3  3  1  2  2  4  6  2  5]
Средний путь до границы: 200.0 см


In [4]:
def generate_directions_scalar(photon_count, seed=123):
    rng = np.random.default_rng(seed)
    directions = []
    for _ in range(photon_count):
        cos_theta = rng.uniform(-1.0, 1.0)
        phi = rng.uniform(0.0, 2 * np.pi)
        sin_theta = sqrt(1.0 - cos_theta**2)
        directions.append((sin_theta * cos(phi), sin_theta * sin(phi), cos_theta))
    return np.asarray(directions)

def propagate_photons_scalar(source_position, directions, detector):
    position = np.asarray(source_position, dtype=float)
    pmt_indices = np.full(len(directions), -1, dtype=np.int64)
    path_lengths = np.empty(len(directions), dtype=float)
    position_radius_squared = float(np.dot(position, position))
    for photon_index, direction in enumerate(directions):
        direction = direction / np.linalg.norm(direction)
        projection = float(np.dot(direction, position))
        path_length = -projection + sqrt(
            projection**2 + detector.R**2 - position_radius_squared
        )
        point = position + path_length * direction
        pmt_projection = detector.pmt_normals @ point
        hit_pmts = np.flatnonzero(pmt_projection >= detector.projection_threshold)
        if len(hit_pmts):
            pmt_indices[photon_index] = hit_pmts[0]
        path_lengths[photon_index] = path_length
    return pmt_indices, path_lengths

def photoelectrons_scalar(pmt_indices, detector, seed=456):
    rng = np.random.default_rng(seed)
    photoelectrons = np.zeros(len(pmt_indices), dtype=np.int64)
    for photon_index, pmt_index in enumerate(pmt_indices):
        if pmt_index >= 0:
            photoelectrons[photon_index] = rng.poisson(lam=detector.a)
    return photoelectrons

def median_runtime(function, repeats=3):
    return float(np.median(timeit.repeat(function, repeat=repeats, number=1)))

In [8]:
benchmark_photon_count = 10000
benchmark_directions = source.generate_directions(
    benchmark_photon_count, rng=np.random.default_rng(2026)
)
benchmark_pmt_indices, _ = detector.propagate_photons(
    source_position, benchmark_directions
)

timings = {
    "Генерация, NumPy": median_runtime(
        lambda: source.generate_directions(benchmark_photon_count, rng=np.random.default_rng(10))
    ),
    "Генерация, цикл": median_runtime(
        lambda: generate_directions_scalar(benchmark_photon_count, seed=10)
    ),
    "Пропагация, NumPy": median_runtime(
        lambda: detector.propagate_photons(source_position, benchmark_directions)
    ),
    "Пропагация, цикл": median_runtime(
        lambda: propagate_photons_scalar(source_position, benchmark_directions, detector)
    ),
    "Фотоэлектроны, NumPy": median_runtime(
        lambda: detector.generate_photoelectrons(
            benchmark_pmt_indices, rng=np.random.default_rng(20)
        )
    ),
    "Фотоэлектроны, цикл": median_runtime(
        lambda: photoelectrons_scalar(benchmark_pmt_indices, detector, seed=20)
    )
}

print(f"Сравнение времени для {benchmark_photon_count:,} фотонов")
print("Медиана 3 запусков; меньшее время означает более быструю обработку.\n")
print(f"{'Этап':<30} {'NumPy, мс':>12} {'Цикл, мс':>12} {'Ускорение':>12}")
print("-" * 70)
stage_labels = {
    "Генерация": "Генерация направлений",
    "Пропагация": "Перенос фотонов",
    "Фотоэлектроны": "Фотоэлектронный отклик"
}
for stage, label in stage_labels.items():
    numpy_time = timings[f"{stage}, NumPy"]
    scalar_time = timings[f"{stage}, цикл"]
    print(
        f"{label:<30} {numpy_time * 1000:>12.3f} "
        f"{scalar_time * 1000:>12.3f} {scalar_time / numpy_time:>11.1f}x"
    )

Сравнение времени для 10,000 фотонов
Медиана 3 запусков; меньшее время означает более быструю обработку.

Этап                              NumPy, мс     Цикл, мс    Ускорение
----------------------------------------------------------------------
Генерация направлений                 0.529       49.041        92.8x
Перенос фотонов                       1.946       90.836        46.7x
Фотоэлектронный отклик                0.153        2.598        17.0x
